# Dixon-Coles goal model (Model 1 of 2)

Predicts each team's goals with a Poisson **attack/defense** model, then (later) refines with Dixon-Coles — the low-score correction and time-decay weighting. This is the model that drives the tournament simulator (it needs scorelines). Elo is the parallel Model 2 in its own notebook.

**Build order here:** load + pre-1970 cut → long-format reshape → base Poisson fit → sanity-check ratings → match prediction (λ → W/D/L) → *(next)* walk-forward evaluation → DC refinements.

In [1]:
import pandas as pd 
import numpy as np
from scipy.stats import poisson
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss

In [2]:
results = pd.read_parquet('../data/interim/results_clean.parquet')

In [3]:
# Data scope: drop pre-1970 (sparse, defunct-team-heavy, near-zero-weight under decay anyway)
# Applied here at modeling time - the cleaned parquet stays a full history
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)

print(f'Before cut: {len(results)}')
print(f'After 1970 cut: {len(model_df)}')
print(f'Dropped: {len(results) - len(model_df)}')
print(f"Date range: {model_df['date'].min().date()} → {model_df['date'].max().date()}")

Before cut: 49482
After 1970 cut: 41522
Dropped: 7960
Date range: 1970-01-04 → 2026-06-30


## Reshape to long format

Each match becomes **two rows** — one per team's scoring perspective (home attacking, away attacking) — so a single Poisson regression can learn every team's attack *and* defense at once. `is_home = 1` only for the home team at non-neutral venues (bakes in the venue decision from EDA).

Wrapped in a reusable **`reshape_to_long`** helper (single source of truth) because the same transform is applied to several frames: the full data (for the sanity fit) and the train slice (for evaluation), and later each walk-forward block. The input frames stay wide; the long output is only ever fitting input.

In [4]:
def reshape_to_long(df):

    home_rows = df.rename(columns={
        'home_team': 'attacking_team',
        'away_team': 'defending_team',
        'home_score': 'goals'
    })[['date', 'attacking_team', 'defending_team', 'goals', 'neutral']].copy()

    home_rows['is_home'] = (~home_rows['neutral']).astype(int) # home advantage only when not neutral

    away_rows = df.rename(columns={
        'away_team': 'attacking_team',
        'home_team': 'defending_team',
        'away_score': 'goals'
    })[['date', 'attacking_team', 'defending_team', 'goals', 'neutral']].copy()

    away_rows['is_home'] = 0 # away side never gets home advantage

    long_df = pd.concat([home_rows, away_rows], ignore_index=True)
    return long_df

long_df = reshape_to_long(model_df)
print(long_df)


            date attacking_team  defending_team  goals  neutral  is_home
0     1970-01-04          Malta      Luxembourg      1    False        1
1     1970-01-14        England     Netherlands      0    False        1
2     1970-01-28         Israel     Netherlands      0    False        1
3     1970-02-04           Peru  Czechoslovakia      0    False        1
4     1970-02-06       Cameroon     Ivory Coast      3     True        0
...          ...            ...             ...    ...      ...      ...
83039 2026-06-29          Japan          Brazil      1     True        0
83040 2026-06-29        Morocco     Netherlands      1     True        0
83041 2026-06-30         Sweden          France      0     True        0
83042 2026-06-30         Norway     Ivory Coast      2     True        0
83043 2026-06-30        Ecuador          Mexico      0    False        0

[83044 rows x 6 columns]


## Base Poisson fit

`goals ~ attacking_team + defending_team + is_home` via `PoissonRegressor`. The fitted coefficients **are** the ratings: attacking-team dummies = attack strength, defending-team dummies = defense strength, `is_home` = home advantage. Pooled fit, no time-split or decay yet — a **sanity fit**, not the evaluation number. `alpha` (L2 penalty) kept small here; tuned properly on walk-forward later.

In [5]:
def fit_poisson_model(df, alpha, sample_weight=None):

    features = df[['attacking_team', 'defending_team', 'is_home']]
    target = df['goals']

    # One-Hot the team columns; is_home passes through untouched
    encoder = ColumnTransformer(
        transformers = [('teams', OneHotEncoder(handle_unknown='ignore'), ['attacking_team', 'defending_team'])],
        remainder='passthrough'
    )

    base_poisson = Pipeline([
        ('encode', encoder),
        ('poisson', PoissonRegressor(alpha=alpha, max_iter=1000))
    ])

    base_poisson.fit(features, target, poisson__sample_weight=sample_weight)
    return base_poisson

base_poisson = fit_poisson_model(long_df, 0.001)
print(base_poisson)

Pipeline(steps=[('encode',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('teams',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['attacking_team',
                                                   'defending_team'])])),
                ('poisson', PoissonRegressor(alpha=0.001, max_iter=1000))])


## Sanity-check the ratings

Pull the learned coefficients out as attack/defense leaderboards. Bar to pass: powerhouses top both lists and `home_adv` ≈ +0.3 (×1.3–1.4, matching EDA). **Result:** home_adv 0.329 ✅, attack/defense both sensible ✅ — but small-sample minnows (e.g. Tahiti on attack) leak in, flagging that regularization needs tuning (deferred to walk-forward, not hand-tuned).

In [6]:
# Extract fitted ratings and align to feature names
encoder = base_poisson.named_steps['encode']
poisson_estimator = base_poisson.named_steps['poisson']

ratings = pd.DataFrame({
    'feature': encoder.get_feature_names_out(),
    'coef': poisson_estimator.coef_
})

# Attack = coefficient on the attacking-team dummy (higher = scores more)
attack = ratings[ratings['feature'].str.contains('attacking_team')].copy()

# Strip the prefix to get a clean team name
attack['team'] = attack['feature'].str.replace('teams__attacking_team_', '', regex=False)

# Sort strongest attack first
attack = attack.sort_values('coef', ascending=False)

# Same for defense
defense = ratings[ratings['feature'].str.contains('defending_team')].copy()
defense['team'] = defense['feature'].str.replace('teams__defending_team_', '', regex=False)
defense = defense.sort_values('coef', ascending=True)

home_adv = ratings.loc[ratings['feature'].str.contains('is_home'), 'coef'].values[0]
print(f'Home advantage coef: {home_adv:.3f} (goals multiplier x{np.exp(home_adv):.2f})')

print("\nTop 10 attack (score most):")
print(attack[['team', 'coef']].head(10).to_string(index=False))

print("\nTop 10 defense (concede least):")
print(defense[['team', 'coef']].head(10).to_string(index=False))

Home advantage coef: 0.329 (goals multiplier x1.39)

Top 10 attack (score most):
       team     coef
     Brazil 0.803021
    Germany 0.738576
  Argentina 0.640803
Netherlands 0.636295
      Spain 0.635423
     France 0.557114
    England 0.546383
     Mexico 0.541317
     Tahiti 0.505302
   Portugal 0.499481

Top 10 defense (concede least):
       team      coef
     Brazil -0.906414
    England -0.854696
     France -0.790409
      Spain -0.787710
      Italy -0.775484
  Argentina -0.757908
Netherlands -0.734389
   Colombia -0.682906
     Russia -0.670233
    Morocco -0.656485


## Predict a match

The reusable core: two λ's (home/away) → Poisson PMF per side → full scoreline matrix (`np.outer`) → sum into W/D/L (`tril` = home win, `trace` = draw, `triu` = away win). Assumes goal independence (plain Poisson) — the Dixon-Coles low-score correction refines this later. Everything downstream (evaluation, simulator, 2026 forecast) calls this.

In [7]:
def predict_match(model, home_team, away_team, neutral=False, max_goals=10):

    # 1. Build the two team-scoring rows the model expects (same shape as training)
    is_home = 0 if neutral else 1
    match_rows = pd.DataFrame({
        'attacking_team': [home_team, away_team],
        'defending_team': [away_team, home_team],
        'is_home': [is_home, 0]
    })

    lam_home, lam_away = model.predict(match_rows) # expected goals for each side

    # 2. Poisson probabilities for 0, 1, 2, ... max_goals for each team
    home_probs = poisson.pmf(np.arange(max_goals + 1), lam_home)
    away_probs = poisson.pmf(np.arange(max_goals + 1), lam_away)

    # 3. Full scoreline matrix: P(home=i, away=j) = home_probs[i] * away_probs[j]
    score_matrix = np.outer(home_probs, away_probs)

    # Sum cells into W/D/L
    home_win = np.tril(score_matrix, -1).sum() # home goals > away goals
    draw = np.trace(score_matrix) # home goals == away goals
    away_win = np.triu(score_matrix, 1).sum() # home goals < away goals

    return {
        'lambda_home': lam_home,
        'lambda_away': lam_away,
        'home_win': home_win,
        'draw': draw,
        'away_win': away_win
    }

result = predict_match(base_poisson, 'Brazil', 'Argentina', neutral=True)
print(f"λ  Brazil {result['lambda_home']:.2f}  |  Argentina {result['lambda_away']:.2f}")
print(f"Brazil win {result['home_win']:.1%}  |  Draw {result['draw']:.1%}  |  Argentina win {result['away_win']:.1%}")

λ  Brazil 1.34  |  Argentina 0.98
Brazil win 45.0%  |  Draw 27.6%  |  Argentina win 27.4%


## Held-out evaluation

To get an honest skill number, split the data leakage-safe:
- **train** — everything before 2024-01-01; the only data the eval model is allowed to see.
- **eval** — 2024 → pre-tournament 2026, **excluding the World Cup itself**; the held-out test set.
- **wc2026** — the World Cup matches, kept aside as the forecast *target*, never in train/eval.

The eval model is **refit on `train` only** — `base_poisson` was fitted on all data (incl. 2024+), so scoring it here would be grading it on its own training set. Boundaries are mutually exclusive (`<` train / `>=` eval) so no match lands in both.

In [8]:
# Leakage-safe split — boundaries mutually exclusive so no match is in two slices
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)

train_df = model_df[model_df['date'] < '2024-01-01']                        # eval model sees only this
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]       # held-out test set (WC excluded)
wc2026_mask_df = model_df[wc2026_mask]                                      # forecast target, held aside

In [9]:
# Create a long df from train_df
long_train = reshape_to_long(train_df)

# Create a model 
train_poisson = fit_poisson_model(long_train, 0.001)

### Ranked Probability Score (RPS)

Ordinal-aware scoring for W/D/L, which is *ordered* by goal difference (home-win → draw → away-win). RPS compares the **cumulative** predicted vs actual distributions, so a near-miss (predicted draw, home won) is penalised less than a far-miss (predicted away-win). Lower = better, averaged over eval matches. The football-standard metric; Brier/log-loss ignore this ordering.

In [10]:
# Ordinal metric: cumsum turns W/D/L into cumulative distributions, so being "close"
# on the ordered scale is rewarded. Divide by (n-1) to normalise to [0, 1].
def ranked_probability_score(predicted_probs, actual_outcome):
    cum_pred = np.cumsum(predicted_probs)
    cum_actual = np.cumsum(actual_outcome)
    n = len(predicted_probs)
    return np.sum((cum_pred - cum_actual) ** 2) / (n-1)

## Low-score correction (Dixon-Coles refinement #2)

Independent Poisson assumes each team scores at its own rate regardless of the other — but real football clusters at low scores (more 0-0 and 1-1 draws than independence predicts). Since draws are one of our three outcomes, that miscalibration hurts RPS.

Dixon-Coles patches exactly the four lowest scorelines by multiplying those cells of the scoreline matrix by a τ factor controlled by one knob **ρ**:

- τ(0,0) = 1 − λ_home·λ_away·ρ
- τ(0,1) = 1 + λ_home·ρ
- τ(1,0) = 1 + λ_away·ρ
- τ(1,1) = 1 − ρ

Every other cell is unchanged. ρ=0 → no correction; larger ρ → more draw mass. We renormalize the matrix afterwards (the tweaks + the goal-tail cut at `max_goals` mean it no longer sums to 1). **ρ is tuned on eval RPS**, holding the best ξ/alpha fixed.

In [20]:
def apply_dc_correction(score_matrix, lambda_home, lambda_away, rho):
    # Multiply the 4 low-score cells by their Dixon-Coles t factors
    score_matrix[0, 0] *= 1 - lambda_home * lambda_away * rho
    score_matrix[0, 1] *= 1 + lambda_home * rho
    score_matrix[1, 0] *= 1 + lambda_away * rho
    score_matrix[1, 1] *= 1 - rho
    return score_matrix / score_matrix.sum()   # renormalize so it sums to 1 again

### `evaluate_model` — batched RPS on held-out data

Scores a fitted model on a set of matches and returns the **mean RPS**. Vectorized for speed: reshape the matches to long format and call `model.predict` **once** — because `reshape_to_long` stacks all home-rows then all away-rows, `preds[:n]` are the home λ's and `preds[n:]` the away λ's (in `eval_df` order). This replaces the old per-match `predict_match` loop (~1000 tiny pipeline calls), which was too slow for the parameter sweeps.

Then per match: build the scoreline matrix → **apply the ρ low-score correction** → collapse to W/D/L → score against the actual result. `rho=0.0` by default (no correction); pass a ρ to switch on Dixon-Coles.

In [22]:
# Create an evaluate_model function
def evaluate_model(model, eval_df, rho=0.0, max_goals=10):
    long_eval = reshape_to_long(eval_df)
    preds = model.predict(long_eval[['attacking_team', 'defending_team', 'is_home']]) 

    n = len(eval_df)
    lambda_home = preds[:n] # home-rows are stacked first
    lambda_away = preds[n:] # away-rows second

    home_score = eval_df['home_score'].to_numpy()
    away_score = eval_df['away_score'].to_numpy()

    rps_values = []
    for i in range(n):
        
        # scoreline -> W/D/L (cheap numpy, no pipeline overhead)
        home_probs = poisson.pmf(np.arange(max_goals + 1), lambda_home[i])
        away_probs = poisson.pmf(np.arange(max_goals + 1), lambda_away[i])
        score_matrix = np.outer(home_probs, away_probs)
        score_matrix = apply_dc_correction(score_matrix, lambda_home[i], lambda_away[i], rho)
        
        probs = [
            np.tril(score_matrix, -1).sum(),
            np.trace(score_matrix),
            np.triu(score_matrix, 1).sum()
        ]

        # One-Hot actual outcome
        if home_score[i] > away_score[i]:
            actual = [1, 0, 0]
        elif home_score[i] == away_score[i]:
            actual = [0, 1, 0]
        else:
            actual = [0, 0, 1]

        value = ranked_probability_score(probs, actual)
        rps_values.append(value)

    # Mean rps
    mean_rps = np.mean(rps_values)

    return mean_rps

print(evaluate_model(train_poisson, eval_df))

0.17819169841671212


In [12]:
# Create naive base rates from train_df
home_rate = (train_df['home_score'] > train_df['away_score']).mean()
draw_rate = (train_df['home_score'] == train_df['away_score']).mean()
away_rate = (train_df['home_score'] < train_df['away_score']).mean()
base_probs = [home_rate, draw_rate, away_rate]

rps_values = []
for _, row in eval_df.iterrows():
    if row['home_score'] > row['away_score']:
        actual = [1, 0, 0]
    elif row['home_score'] == row['away_score']:
        actual = [0, 1, 0]
    else:
        actual = [0, 0, 1]
    
    value = ranked_probability_score(base_probs, actual)
    rps_values.append(value)

print(np.mean(rps_values))

0.22745510855113082


## Time-decay weighting (Dixon-Coles refinement #1)

Weight each training match by `exp(-ξ·age)` so recent form counts more and old matches fade smoothly — the recency decision from EDA, done properly. Age is in years relative to the train cutoff; **ξ** is the decay rate (higher = forget faster), **tuned on eval RPS** (jointly with `alpha` later). It flows into the fit via `poisson__sample_weight`.

Sanity: ξ=0.001/yr is near-zero decay, so RPS barely moves (0.17820 vs 0.17831 unweighted) — confirms the wiring. The real gain comes from **sweeping ξ** to find the decay that minimises eval RPS.

In [13]:
# Time-decay weights: recent matches count more, old ones fade (exponential decay)
xi = 0.001                                                            # decay rate per year — to be tuned on eval RPS
reference_date = long_train['date'].max()                            # as-of point: newest train match
age_years = (reference_date - long_train['date']).dt.days / 365.25   # each match's age in years
sample_weight = np.exp(-xi * age_years)                              # weight in (0, 1], newest ≈ 1

# Refit train model with the decay weights, then score on held-out eval
train_poisson = fit_poisson_model(long_train, 0.001, sample_weight)
print(evaluate_model(train_poisson, eval_df))

0.17820203282198965


In [19]:
# Joint sweep of decay (ξ) and regularization (alpha) — pick the pair with lowest eval RPS
xis = [0.05, 0.1, 0.2, 0.3, 0.5]
alphas = [0.00001, 0.00005, 0.0001, 0.0005]

# Initialize rps and params
best_rps = 100
best_params = (1, 1)

for xi in xis:
    # Create sample weight 
    sample_weight = np.exp(-xi * age_years)

    for alpha in alphas:
        # Fit & predict
        model = fit_poisson_model(long_train, alpha, sample_weight=sample_weight)
        rps = evaluate_model(model, eval_df)
        print(f"xi={xi}  alpha={alpha}  rps={rps:.5f}")

        # Best rps and params
        if rps < best_rps:
            best_rps = rps
            best_params = (xi, alpha)

print(f'Best RPS: {best_rps:.4f} with {best_params}')

xi=0.05  alpha=1e-05  rps=0.16878
xi=0.05  alpha=5e-05  rps=0.16880
xi=0.05  alpha=0.0001  rps=0.16890
xi=0.05  alpha=0.0005  rps=0.17080
xi=0.1  alpha=1e-05  rps=0.16658
xi=0.1  alpha=5e-05  rps=0.16664
xi=0.1  alpha=0.0001  rps=0.16678
xi=0.1  alpha=0.0005  rps=0.16886
xi=0.2  alpha=1e-05  rps=0.16529
xi=0.2  alpha=5e-05  rps=0.16535
xi=0.2  alpha=0.0001  rps=0.16557
xi=0.2  alpha=0.0005  rps=0.16804
xi=0.3  alpha=1e-05  rps=0.16542
xi=0.3  alpha=5e-05  rps=0.16555
xi=0.3  alpha=0.0001  rps=0.16580
xi=0.3  alpha=0.0005  rps=0.16854
xi=0.5  alpha=1e-05  rps=0.16700
xi=0.5  alpha=5e-05  rps=0.16719
xi=0.5  alpha=0.0001  rps=0.16741
xi=0.5  alpha=0.0005  rps=0.17058
Best RPS: 0.1653 with (0.2, 1e-05)
